# ZoeDepth NYU+KITTI — DIMER monocular metric depth estimation tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/zoedepth-metric-depth-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/zoedepth-metric-depth-pipeline/blob/main/tutorials/zoedepth_metric_depth_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Intel%2Fzoedepth--nyu--kitti-ffcc4d?style=flat)](https://huggingface.co/Intel/zoedepth-nyu-kitti) [![Upstream](https://img.shields.io/badge/Upstream-isl--org%2FZoeDepth-181717?style=flat&logo=github&logoColor=white)](https://github.com/isl-org/ZoeDepth) [![arXiv](https://img.shields.io/badge/arXiv-2302.12288-b31b1b.svg)](https://arxiv.org/abs/2302.12288)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** monocular metric depth estimation — one RGB image → a float32 depth map in metres at the input resolution — using the pinned `Intel/zoedepth-nyu-kitti` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/zoedepth_metric_depth_pipeline/pipeline.py` at revision `b701b5c40d70`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `f364d4c7936e91f465abba182208dd68142bf0ca` (~1380 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the ZoeDepth model (a BEiT-large DPT encoder–decoder for relative depth, 24 layers, hidden size 1024, with two metric bin heads — NYU indoor, 0.001–10 m, and KITTI outdoor, 0.001–80 m — chosen per image by a latent domain classifier; about 345M parameters) turns one RGB image into a depth map in metres, which the carried module interpolates back to the input resolution through the pinned processor's post-processing; an optional horizontal-flip test-time augmentation (the upstream evaluation setting) averages two passes. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights and processor, and the carried module adds snapshot verification, the input contract (side ceilings, aspect-ratio ceiling, a boolean flip flag), a fixed output contract (metres, float32, minimum/median/maximum), and the `abs_rel`, `delta1`, `validate_inputs` and `evaluation_report` helpers. The default sample is a flat cartoon room drawn in code with **no reference depth**, so the evaluation report is `not-measurable` by design (the fleet matrix scores this row only with a reference) and the printed ordering check — is the near box estimated nearer than the far cabinet? — is an observation, not a metric.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, draw a synthetic room (or upload your own photograph and, optionally, a metric reference depth map) and validate it into an input manifest, choose whether to use flip augmentation, run the supported task, read the depth map correctly (metres, but an estimate with no confidence and a scale that depends on the model recognising the scene), exercise an optional BYOD path, produce an evaluation report that is `sample-sanity` with `abs_rel` and `delta1` only when a reference exists and `not-measurable` otherwise, and export the depth array, a preview and provenance.

**This notebook does not demonstrate:** Relative or affine-invariant depth (this checkpoint claims metres; for scale-free depth see the sibling Depth Anything pipeline), camera intrinsics or point-cloud reconstruction (a depth map is not a 3D model without them), video or multi-view consistency, batch throughput, evaluation on NYU Depth v2 or KITTI (not bundled; only a drawn room is run here, and it is scored only if you bring a reference), and any training. The model was fine-tuned on indoor NYU and outdoor KITTI photographs; flat drawings, documents, medical, aerial and underwater imagery and unusual cameras are outside what this notebook measures, and a smooth-looking depth map carries no signal.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. CPU is adequate: the repository's model card records 5.3 s to load and 1.5 s per 640×480 image (2.7 s with flip augmentation) in the Windows venv (Intel Core Ultra 9 275HX). The pinned `torch==2.14.0` install and the 1.38 GB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python, NumPy and PIL; what metric depth in metres means and why a monocular estimate of it depends on the model guessing the scene scale; what absolute relative error and δ1 measure and why one drawn image is not a benchmark.
- **Data:** the default sample is a deterministic 640×480 cartoon room drawn in code with Pillow (a wall, a floor, a far cabinet on the wall, a near box on the floor and a window; no text rendering, so its digest is stable across Pillow builds) with **no reference depth**, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one image decodable by Pillow (PNG/JPEG/WebP and similar), any colour mode, sides between 32 and 4096 px, aspect ratio at most 4:1, plus optionally a `.npy` float array of metric depth in metres with the same height and width (non-positive pixels are ignored). Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `Intel/zoedepth-nyu-kitti` snapshot (~1380 MB in total) at revision `f364d4c7936e…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'zoedepth-metric-depth-pipeline',
    'repository_revision': 'b701b5c40d706470a3d6afbde9b22adb6ee40747',
    'embedded_module': 'src/zoedepth_metric_depth_pipeline/pipeline.py',
    'embedded_modules': ['src/zoedepth_metric_depth_pipeline/pipeline.py'],
    'module_sha256': '6883cbf85c9ac398c57d2ed4cd9f4bf5ebb7936ad1242ad20ec7e26e98cb2fda',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/zoedepth_metric_depth_pipeline/` @ `b701b5c40d70`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/zoedepth_metric_depth_pipeline/pipeline.py`

In [ ]:
"""Monocular metric depth estimation with the pinned ``Intel/zoedepth-nyu-kitti`` checkpoint (ZoeDepth).

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the ZoeDepth architecture (a BEiT-large DPT backbone with metric bin heads
for the NYU and KITTI ranges) comes from the pinned ``transformers`` release, the weights are
SafeTensors, and no model-repository code is executed. The output is depth in metres — a metric
estimate with no confidence, not a measurement — at the caller's resolution.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

MODEL_ID = "Intel/zoedepth-nyu-kitti"
MODEL_REVISION = "f364d4c7936e91f465abba182208dd68142bf0ca"
MODEL_LICENSE = "mit"
MODEL_KEY = "zoedepth-nyu-kitti"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Input ceilings. The ZoeDepth processor resizes the image to fit 384x512 while keeping the aspect
# ratio (sides rounded to multiples of 32) and pads, so the backbone cost grows with the aspect ratio,
# not the pixel count; the prediction is interpolated back to the caller's resolution.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 32
MAX_ASPECT_RATIO = 4.0
DEPTH_KIND = "metric"
DEPTH_UNIT = "metres"
# The checkpoint's two metric heads (config.json bin_configurations): NYU indoor 0.001-10 m and KITTI
# outdoor 0.001-80 m; the model routes each image to one head by its own domain classifier.
DEPTH_RANGES_M = {"nyu": (0.001, 10.0), "kitti": (0.001, 80.0)}
# Optional horizontal-flip test-time augmentation (the upstream evaluation setting): two forward
# passes, predictions averaged. Off by default; a caller-owned request parameter.
FLIP_AUGMENTATION = False
# delta1 accuracy threshold (the depth-estimation convention): max(pred/ref, ref/pred) < 1.25.
DELTA_THRESHOLD = 1.25


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _valid_mask(pred: np.ndarray, ref_depth: np.ndarray) -> np.ndarray:
    if pred.shape != ref_depth.shape:
        raise ValueError(f"shape mismatch: pred {pred.shape} vs ref {ref_depth.shape}")
    valid = np.isfinite(ref_depth) & (ref_depth > 0) & np.isfinite(pred) & (pred > 0)
    if valid.sum() < 2:
        raise ValueError("need at least 2 valid reference pixels (ref_depth > 0 and pred > 0)")
    return valid


def abs_rel(pred: np.ndarray, ref_depth: np.ndarray) -> float:
    """Absolute relative error ``mean(|pred - ref| / ref)`` of metric depth against metric reference depth.

    Both arrays are in metres at the same H x W; no scale or shift alignment is applied, because the
    model claims metric output — a scale error therefore shows up in the number, as it should.
    """
    pred = np.asarray(pred, dtype=np.float64)
    ref_depth = np.asarray(ref_depth, dtype=np.float64)
    valid = _valid_mask(pred, ref_depth)
    return float(np.mean(np.abs(pred[valid] - ref_depth[valid]) / ref_depth[valid]))


def delta1(pred: np.ndarray, ref_depth: np.ndarray, *, threshold: float = DELTA_THRESHOLD) -> float:
    """Fraction of valid pixels whose ratio ``max(pred/ref, ref/pred)`` is below ``threshold`` (1.25)."""
    pred = np.asarray(pred, dtype=np.float64)
    ref_depth = np.asarray(ref_depth, dtype=np.float64)
    valid = _valid_mask(pred, ref_depth)
    ratio = np.maximum(pred[valid] / ref_depth[valid], ref_depth[valid] / pred[valid])
    return float(np.mean(ratio < threshold))


def validate_image(image: Any) -> Image.Image:
    """Type- and size-check a caller image and return it as RGB."""
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    short, long = min(width, height), max(width, height)
    if short < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {short} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if long > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {long} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    if long / short > MAX_ASPECT_RATIO:
        raise ValueError(f"aspect ratio {long / short:.2f} > MAX_ASPECT_RATIO {MAX_ASPECT_RATIO}")
    return image.convert("RGB")


def _check_flip(value: Any) -> bool:
    if not isinstance(value, bool):
        raise TypeError("flip_augmentation must be a bool")
    return value


INPUT_SCHEMA: dict[str, Any] = {
    "input": "PIL.Image.Image, or a sequence of them for the validation stage; any mode, converted to RGB",
    "short_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "long_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "aspect_ratio": [1.0, MAX_ASPECT_RATIO],
    "flip_augmentation": "bool; two forward passes (image and its mirror) averaged when true",
    "output": (
        f"{DEPTH_KIND} depth in {DEPTH_UNIT}, float32 H x W at the input resolution (larger = farther); "
        "an estimate with no confidence, routed to the NYU (0.001-10 m) or KITTI (0.001-80 m) head by "
        "the model's own domain classifier"
    ),
    "preprocessing": (
        "convert to RGB; the ZoeDepth processor resizes to fit 384x512 keeping the aspect ratio "
        "(sides rounded to multiples of 32), pads, normalises with mean/std 0.5; the prediction is "
        "interpolated back to the input resolution by post_process_depth_estimation"
    ),
}


def validate_inputs(
    images: Any, *, flip_augmentation: bool = FLIP_AUGMENTATION, names: Sequence[str] | None = None
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, request, verdict).

    Each image is routed through the public ``validate_image`` that ``predict`` itself calls, so a
    rejection here raises exactly what ``predict`` would; a caller that wants the finding recorded
    catches the exception and stores ``str(exc)`` under ``findings``.
    """
    batch = [images] if isinstance(images, Image.Image) else images
    if not isinstance(batch, Sequence) or isinstance(batch, str | bytes):
        raise TypeError("images must be a PIL.Image.Image or a sequence of them")
    if len(batch) < 1:
        raise ValueError("at least one image is required")
    if names is not None and len(names) != len(batch):
        raise ValueError("names must have one entry per image")
    flip = _check_flip(flip_augmentation)
    inputs = []
    for index, candidate in enumerate(batch):
        rgb = validate_image(candidate)
        width, height = rgb.size
        long_side, short_side = max(width, height), min(width, height)
        inputs.append(
            {
                "id": names[index] if names else f"image-{index}",
                "mode": getattr(candidate, "mode", rgb.mode),
                "size": [width, height],
                "aspect_ratio": round(long_side / short_side, 3),
            }
        )
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": inputs,
        "n_images": len(inputs),
        "flip_augmentation": flip,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    reference_depth: Any | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``reference_depth`` (metric depth in metres, same H x W as the prediction, non-positive or
    non-finite pixels ignored) the report carries ``abs_rel`` and ``delta1`` computed without any
    alignment — the model claims metres, so scale errors count — as sample-sanity evidence. Without it
    the verdict is ``not-measurable`` and the report says what ground truth would make the task
    measurable: a depth map in metres has no intrinsic score.
    """
    depth = np.asarray(result["depth"])
    base = {
        "task": "monocular metric depth estimation",
        "score_semantics": (
            f"{result.get('depth_kind', DEPTH_KIND)} depth in {DEPTH_UNIT} with no confidence; the value "
            "is an estimate whose scale depends on the model having recognised the scene's domain and "
            "camera, and a blank image still yields a depth map"
        ),
        "sample_kind": sample_kind,
        "flip_augmentation": bool(result.get("flip_augmentation", FLIP_AUGMENTATION)),
        "n_images": 1,
        "n_pixels": int(depth.size),
        "depth_min_m": float(depth.min()),
        "depth_max_m": float(depth.max()),
        "depth_median_m": float(np.median(depth)),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if reference_depth is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no metric reference depth was supplied for the evaluated image",
            "needs": (
                "a metric depth map in metres with the same height and width as the image, from a depth "
                "sensor, LiDAR, stereo or an RGB-D benchmark, scored with abs_rel and delta1 against a "
                "constant-depth prior (the reference's median) as the trivial baseline"
            ),
        }
    ref = np.asarray(reference_depth, dtype=np.float64)
    valid = np.isfinite(ref) & (ref > 0)
    n_valid = int(valid.sum())
    constant = np.full_like(ref, float(np.median(ref[valid])) if n_valid else 1.0)
    return {
        **base,
        "metrics": [
            {
                "id": "abs_rel",
                "value": abs_rel(depth, ref),
                "align": False,
                "n_valid_pixels": n_valid,
                "estimation": "single image, no alignment (metric output as-is), no dispersion estimate",
            },
            {
                "id": "delta1",
                "value": delta1(depth, ref),
                "threshold": DELTA_THRESHOLD,
                "n_valid_pixels": n_valid,
                "estimation": "single image, fraction of valid pixels within the ratio threshold",
            },
        ],
        "baselines": [
            {
                "id": "constant_median_depth",
                "abs_rel": abs_rel(constant, ref),
                "delta1": delta1(constant, ref),
                "note": "every pixel predicted at the reference's median depth",
            }
        ],
        "verdict": "sample-sanity",
        "reason": "one image with caller-supplied metric depth; not a benchmark",
        "needs": "a held-out set of metric depth maps from the deployment domain for any generalisable claim",
    }


@dataclass
class ZoeDepthMetricPipeline:
    """Monocular metric depth estimation over the pinned ZoeDepth NYU+KITTI checkpoint."""

    _runner: Callable[[Image.Image, bool], np.ndarray]
    device: str

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> ZoeDepthMetricPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import ZoeDepthForDepthEstimation, ZoeDepthImageProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = ZoeDepthImageProcessor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = ZoeDepthForDepthEstimation.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, dtype=torch.float32, **kwargs
        )
        model = model.to(resolved_device).eval()

        def runner(image: Image.Image, flip: bool) -> np.ndarray:
            inputs = processor(images=image, return_tensors="pt").to(resolved_device)
            with torch.inference_mode():
                outputs = model(**inputs)
                extra = {}
                if flip:
                    mirrored = torch.flip(inputs["pixel_values"], dims=[3])
                    extra["outputs_flipped"] = model(pixel_values=mirrored)
            # The pinned processor's post-processing un-pads, un-flips (when given) and interpolates
            # the prediction back to the source size; it returns metres.
            post = processor.post_process_depth_estimation(
                outputs, source_sizes=[(image.height, image.width)], **extra
            )
            return post[0]["predicted_depth"].float().cpu().numpy()

        return cls(runner, resolved_device)

    def predict(self, image: Image.Image, *, flip_augmentation: bool = FLIP_AUGMENTATION) -> dict[str, Any]:
        """Return metric depth in metres as a float32 H x W array at the input resolution."""
        rgb = validate_image(image)
        flip = _check_flip(flip_augmentation)
        depth = np.asarray(self._runner(rgb, flip), dtype=np.float32)
        if depth.shape != (rgb.height, rgb.width):
            raise RuntimeError(f"backend returned shape {depth.shape}, expected {(rgb.height, rgb.width)}")
        if not np.all(np.isfinite(depth)) or depth.min() <= 0:
            raise RuntimeError("backend returned non-finite or non-positive depth values")
        return {
            "depth": depth,
            "depth_kind": DEPTH_KIND,
            "depth_unit": DEPTH_UNIT,
            "depth_min": float(depth.min()),
            "depth_max": float(depth.max()),
            "depth_median": float(np.median(depth)),
            "flip_augmentation": flip,
            "height": rgb.height,
            "width": rgb.width,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `f364d4c7936e…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `ZoeDepthMetricPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "zoedepth-nyu-kitti",
  "modelId": "Intel/zoedepth-nyu-kitti",
  "revision": "f364d4c7936e91f465abba182208dd68142bf0ca",
  "files": [
    {
      "path": "README.md",
      "bytes": 2508,
      "sha256": "3a24f725d6ba1ae7144a50b08eca542965bd99d4c003d2898e77db56cdb6ca9a"
    },
    {
      "path": "config.json",
      "bytes": 2225,
      "sha256": "58494c160c520023c4d5bdeebb3b2d035e37e48cc81225580f49fcb06e175913"
    },
    {
      "path": "model.safetensors",
      "bytes": 1380374404,
      "sha256": "c5494fa0938f18d71e215e245472470c3aefebd7b434abd89750e5ae4008e2dc"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 723,
      "sha256": "0b64d8edc980d7b7abb819085650c43da8b8183acbd4336ab5a9e0e9caf4648e"
    }
  ],
  "totalBytes": 1380379860
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = ZoeDepthMetricPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Draw the synthetic room or optional BYOD

The default sample is **synthetic**: a flat cartoon room — a grey wall, a brown floor meeting it at a visible edge, a blue cabinet high on the wall, a red box low on the floor and a pale window — is drawn with Pillow at 640×480, the same drawing the repository's smoke run used. It has **no reference depth**: a drawing has no metres in it, and the notebook does not invent any, so the evaluation report will be `not-measurable` and the only check is an ordering observation (the box, drawn low and large, should come out nearer than the cabinet, drawn high and small). The image digest is printed for the record. BYOD is optional and disabled by default; when enabled, upload one photograph and, if you have one, a `.npy` depth map in metres with the same height and width — then the report becomes `sample-sanity` with `abs_rel` and `delta1`.

Flip augmentation is a **caller-owned request parameter**: `flip_augmentation` runs the image and its mirror and averages the two depth maps (the upstream evaluation setting; twice the cost, and the smoke run saw a mean difference of 0.024 m on this room). Nothing is validated in this cell — the next section hands the image to the pipeline's own validation stage, which is the only checker. Look for a dictionary naming the sample kind, the image size and digest, the flip flag and whether a reference exists.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw

USE_BYOD = False  # @param {type:"boolean"}
flip_augmentation = False  # @param {type:"boolean"}


def synthetic_room(width=640, height=480):
    """A flat cartoon room drawn with Pillow (no text): wall, floor, far cabinet, near box, window."""
    image = Image.new('RGB', (width, height), (200, 200, 190))  # wall
    d = ImageDraw.Draw(image)
    d.rectangle([0, 300, width, height], fill=(140, 110, 80))  # floor
    d.polygon([(0, 300), (width, 300), (width, 320), (0, 320)], fill=(120, 95, 70))  # skirting edge
    d.rectangle([80, 120, 200, 260], fill=(90, 120, 200))  # far cabinet, high on the wall
    d.rectangle([380, 250, 560, 420], fill=(200, 60, 60))  # near box, low on the floor
    d.rectangle([390, 260, 550, 300], fill=(230, 100, 100))  # box lid
    d.ellipse([250, 60, 330, 140], fill=(255, 240, 150))  # window / light
    probes = {'near box': (470, 340), 'far cabinet': (140, 190), 'floor near the camera': (320, 470), 'wall top': (320, 30)}
    return image, probes


reference_depth = None
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(name for name in uploaded if not name.lower().endswith('.npy'))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    depth_files = [name for name in uploaded if name.lower().endswith('.npy')]
    if depth_files:
        reference_depth = np.load(io.BytesIO(uploaded[depth_files[0]])).astype(np.float64)  # metres, H x W
    probes = {}
    sample_kind = 'BYOD'
else:
    # Deterministic drawing: no randomness and no text rendering, so no seed is needed and the digest is stable.
    image, probes = synthetic_room()
    image_name = 'synthetic_room_640x480.png'
    sample_kind = 'synthetic'

image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'mode': image.mode, 'size': image.size, 'rgb_sha256': image_sha256, 'flip_augmentation': flip_augmentation, 'has_reference_depth': reference_depth is not None})

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `predict` applies — image type, sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px, aspect ratio at most `MAX_ASPECT_RATIO`, and a boolean flip flag — and returns an **input manifest** naming the schema (including the 384×512 aspect-preserving resize, the padding and the post-processing), the input's observed mode, size and aspect ratio, the flip flag and the verdict. The manifest is written to `outputs/zoedepth_metric_depth_input_manifest.json`. To show what rejection looks like, the cell also validates a 5:1 panorama and records the pipeline's own error message as a finding. Inside the pipeline the image is converted to RGB and resized to fit 384×512 with its aspect ratio kept; nothing else is dropped or altered. The pipeline cannot tell whether the image is a photograph, which camera took it, or whether the scene is indoors or outdoors — the model guesses the last from the image, and that guess sets the metres.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_ASPECT_RATIO': MAX_ASPECT_RATIO, 'DEPTH_KIND': DEPTH_KIND, 'DEPTH_UNIT': DEPTH_UNIT, 'DEPTH_RANGES_M': DEPTH_RANGES_M, 'FLIP_AUGMENTATION': FLIP_AUGMENTATION, 'DELTA_THRESHOLD': DELTA_THRESHOLD}})
input_manifest = validate_inputs(image, flip_augmentation=flip_augmentation, names=[image_name])
# Demonstrate rejection on a request that breaks the contract; the finding is recorded, not swallowed.
try:
    validate_inputs(Image.new('RGB', (2000, 400)))
except ValueError as exc:
    input_manifest['findings'].append({'input': 'panorama-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/zoedepth_metric_depth_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Estimate depth and read the output correctly

`predict` returns `depth` (a float32 H×W array in metres at the input resolution; larger is farther), `depth_kind` and `depth_unit`, the minimum, median and maximum, the flip flag, the image size and the model identity. **The metres are an estimate, not a measurement**: the model picks its indoor or outdoor head from the image itself, the scale follows that choice and the camera it assumes, no confidence is attached, and a blank image still yields a depth map. The forward pass is deterministic on a fixed device and dtype; CUDA kernels can shift values slightly, so GPU and CPU maps need not match to the millimetre. Each call costs one BEiT-large pass at up to 384×512 (about 1.5 s on the reference CPU, 2.7 s with flip). As recorded in the model card, the repository's CPU smoke on this same room estimated 1.59–1.90 m with the near box at 1.69 m and the far cabinet at 1.87 m — the right ordering — and estimated 1.43–2.33 m for a blank white image and 1.39–1.82 m for uniform noise: the model always produces plausible-looking metres. The cell prints the depth at the drawn probe points as an observation.

In [ ]:
import time

t0 = time.time()
result = pipe.predict(image, flip_augmentation=flip_augmentation)
elapsed = round(time.time() - t0, 2)
depth = result['depth']
print({'device': pipe.device, 'seconds': elapsed, 'shape': depth.shape, 'dtype': str(depth.dtype), 'unit': result['depth_unit'], 'min_m': round(result['depth_min'], 3), 'median_m': round(result['depth_median'], 3), 'max_m': round(result['depth_max'], 3), 'flip_augmentation': result['flip_augmentation']})
probe_depths = {name: round(float(depth[y, x]), 3) for name, (x, y) in probes.items()}
if probe_depths:
    print('depth at drawn probes (m):', probe_depths)
    print('near box nearer than far cabinet:', probe_depths['near box'] < probe_depths['far cabinet'])

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. No accuracy is reported by default: metric depth needs a reference depth map in metres from a sensor, LiDAR, stereo or an RGB-D benchmark, and this repository ships none (NYU Depth v2 and KITTI are not bundled). When a reference is supplied the report carries `abs_rel` (mean |pred − ref| / ref) and `delta1` (the fraction of pixels whose ratio is within 1.25), both computed **without any scale or shift alignment** because the model claims metres, plus a constant-median-depth baseline, with the verdict `sample-sanity`. On the synthetic path no reference exists — a drawing has no metres — so the verdict is `not-measurable` by design and the report states what would make the task measurable; the probe ordering from the previous section is attached to the report file under `observations` for the record. The report is written to `outputs/zoedepth_metric_depth_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, reference_depth, sample_kind=sample_kind)
report['observations'] = {'probe_depths_m': probe_depths}
with open('outputs/zoedepth_metric_depth_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in report.items() if k not in ('metrics', 'baselines', 'observations')}, indent=2))
for metric in report['metrics']:
    print(f"{metric['id']:10} {metric['value']:.4f}  ({metric['estimation']})")
for baseline in report['baselines']:
    print(f"baseline {baseline['id']}: abs_rel {baseline['abs_rel']:.4f}, delta1 {baseline['delta1']:.4f}  ({baseline['note']})")
if report['verdict'] == 'not-measurable':
    print('No metric reference depth exists for this image, so nothing is scored; the metres are an unverified estimate.')

## 8. Export outputs and provenance

Machine-readable JSON preserves the depth statistics, the flip flag, the evaluation report with the probe observations, the input manifest, the sample identity and digest, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, device); the depth map itself is written as a float32 `.npy` in metres (the array intended for downstream use), because arrays do not belong in JSON. A side-by-side preview PNG shows the image next to the depth map rendered on a fixed grey ramp between its own minimum and maximum (a supplement to, not a replacement for, the array — the ramp is per-image and says nothing about absolute scale). No credentials are recorded.

In [ ]:
np.save('outputs/zoedepth_metric_depth_depth.npy', depth)
lo, hi = float(depth.min()), float(depth.max())
ramp = ((depth - lo) / max(hi - lo, 1e-6) * 255.0).round().astype(np.uint8)
depth_preview = Image.fromarray(ramp).convert('RGB')
preview = Image.new('RGB', (image.width * 2, image.height), 'white')
preview.paste(image.convert('RGB'), (0, 0))
preview.paste(depth_preview, (image.width, 0))
preview.save('outputs/zoedepth_metric_depth_preview.png')
payload = {
    'prediction': {k: v for k, v in result.items() if k != 'depth'},
    'depth_file': 'outputs/zoedepth_metric_depth_depth.npy',
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'has_reference_depth': reference_depth is not None},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/zoedepth_metric_depth_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The depth map is the model's estimate of metres for an image whose camera and scene it has never seen; nothing in the output scores that estimate, the scale follows the model's own indoor/outdoor guess, and it produces metres for any input — a blank white image came out at 1.4–2.3 m in the smoke run. On the drawn room the evaluation report is `not-measurable` by design and the probe ordering (near box 1.69 m before far cabinet 1.87 m in the smoke run) is an observation about a flat cartoon, not evidence of metric accuracy; it says nothing about photographs, cameras with unusual focal lengths, outdoor scale, reflective or transparent surfaces, thin structures, or anything beyond 10 m indoors and 80 m outdoors, and a BYOD result without a reference is a single-image observation with the same verdict. **A smooth depth map is not a correct one**: bring a metric reference (a sensor, LiDAR, stereo or an RGB-D benchmark) and read `abs_rel` and `delta1` against the constant-median baseline before trusting any number. The pipeline provides no camera intrinsics, no point cloud, no confidence, no benchmark evaluation and no training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated request, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** set `flip_augmentation` to `True` and compare the two maps; move the red box up the wall in `synthetic_room` and watch its estimated depth grow; enable `USE_BYOD` with an indoor photograph, then an outdoor one, and compare the ranges the model chose; if you own an RGB-D capture, upload its depth as a `.npy` in metres and see the verdict switch to `sample-sanity`.

## References

- Repository README: https://github.com/kurtvalcorza/zoedepth-metric-depth-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/zoedepth-metric-depth-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/zoedepth-metric-depth-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/Intel/zoedepth-nyu-kitti
- Upstream code: https://github.com/isl-org/ZoeDepth
- ZoeDepth: Zero-shot Transfer by Combining Relative and Metric Depth (Bhat et al., 2023): https://arxiv.org/abs/2302.12288
- Vision Transformers for Dense Prediction — DPT (Ranftl, Bochkovskiy, Koltun, 2021): https://arxiv.org/abs/2103.13413
- Indoor Segmentation and Support Inference from RGBD Images — NYU Depth v2 (Silberman et al., 2012): https://cs.nyu.edu/~fergus/datasets/nyu_depth_v2.html